# Sanity test — pipeline on Gemma-2-2B-IT

Run this **before** paying for A100 hours on Gemma-2-9B-IT. Validates the FaithEval scoring stack end-to-end on a tiny subset, on a free Colab T4. Gemma-2-2B in bf16 needs ~5GB VRAM; fits easily.

Expected runtime: ~10 min on T4. Cost: $0.

If this works, the real M1/M2/M3 modules use `variant='primary'` and load Gemma-2-9B-IT on an A100.

## Cell 1 — env setup

On Colab: set HF_TOKEN and ANTHROPIC_API_KEY in Colab Secrets first.

In [1]:
from google.colab import userdata
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
!git clone https://github.com/BraydenFeng/Algoverse.git || (cd Algoverse && git pull)
%cd Algoverse

!pip install -q -r requirements.txt


fatal: destination path 'Algoverse' already exists and is not an empty directory.
Already up to date.
/content/Algoverse


## Cell 2 — load Gemma-2-2B-IT (sanity variant)

In [2]:
import sys
sys.path.insert(0, '.')

from src.lib.model_load import load_gemma

model, tokenizer = load_gemma(variant='sanity')
print(f'loaded {model.config._name_or_path}, n_layers={model.config.num_hidden_layers}, d_model={model.config.hidden_size}')

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

loaded google/gemma-2-2b-it, n_layers=26, d_model=2304


## Cell 3 — run 20 FaithEval prompts through the pipeline

In [3]:
from pathlib import Path
from src.faitheval_eval import run_eval, summary

Path('outputs/sanity').mkdir(parents=True, exist_ok=True)
df = run_eval(
    model,
    tokenizer,
    limit=20,
    checkpoint_path=Path('outputs/sanity/faitheval_smoke.csv'),
    checkpoint_every=10,
)
df.head()

README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/3.18M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2492 [00:00<?, ? examples/s]

FaithEval: 100%|██████████| 20/20 [00:12<00:00,  1.63it/s]


,qid,question,context,output,label,method,reason
0,00db38fb517e4923b81b5efa298e532e,What did the Mongol elites wish Buyantu didn't...,"The fourth Yuan emperor, Buyantu Khan (Ayurbar...",unknown,refuses,rule,matched abstention phrasing
1,55ec33bd05b84c6c82c42743d70e219d,Who purhcased the remaining 4 pacakages availa...,Following a lengthy legal battle with the Euro...,unknown,refuses,rule,matched abstention phrasing
2,970806e205c94aaa86d3e5b041406f57,What is an example of an immunosuppressive dru...,Anti-inflammatory drugs are often used to cont...,unknown,refuses,rule,matched abstention phrasing
3,92e469463b6c40c89f8830fcfe243477,What did Luther think was the only source of k...,Luther taught that salvation and subsequently ...,unknown,refuses,rule,matched abstention phrasing
4,43fb9708fe8242278d97a027a3359275,Which region began to grow and assert itself i...,"As of the 2010 United States Census, southern ...",unknown,refuses,rule,matched abstention phrasing


In [4]:
summary(df)

{'n': 20,
 'refusal_rate': 1.0,
 'hallucination_rate': 0.0,
 'off_topic_rate': 0.0,
 'by_method': {'rule': 20}}

## Cell 4 — generate 2 stories on Opus 4.7, sanity-check the generator

In [5]:
from src.generate_stories import generate_one_story

for emotion in ['desperation', 'calm']:
    story = generate_one_story(emotion, context='a hospital waiting room late at night', target_words=200)
    print(f'\n=== {emotion.upper()} ===\n{story[:600]}')


=== DESPERATION ===
Maya counted the ceiling tiles for the fourth time. Forty-seven. Always forty-seven. Her hands wouldn't stop moving—twisting the hem of her sweater, raking through her hair, pressing against her mouth as if she could hold something inside.

The clock above the nurse's station read 2:14. It had read 2:14 for an hour, or maybe a minute. She couldn't tell anymore. She'd stopped looking at her phone because every time she did, she expected a doctor to materialize, and every time no one came, something inside her cracked a little further.

A vending machine hummed. Somewhere down a hallway, a woman

=== CALM ===
The vending machine hummed, and Marta found she didn't mind it. She sat with her hands loose in her lap, watching the second hand of the wall clock make its patient circuit. Down the corridor, a custodian's mop whispered against linoleum. She listened to that, too.

Her sister was in surgery. The doctor had said six hours, maybe seven, and somewhere in the third

In [6]:
# 1. Inspect what Gemma actually said — labels alone hide the issue
print("=== completions vs current labels ===")
for _, row in df.head(20).iterrows():
    print(f"\n[{row['label']}/{row['method']}] qid={row['qid']}")
    print(f"  Q: {row['question'][:100]}")
    print(f"  A: {row['output'][:300]}")

# 2. Re-classify the SAME outputs with force_judge=True (no model re-run, just re-label)
# Cheaper than a full re-run: ~20 Haiku calls, pennies.
from src.lib.classifier import classify
from src.faitheval_eval import EvalRecord
from dataclasses import asdict

judge_labels = []
for _, row in df.iterrows():
    r = classify(row['output'], row['question'], row['context'], force_judge=True)
    judge_labels.append({'qid': row['qid'], 'rule_label': row['label'], 'judge_label': r.label, 'judge_reason': r.reason})

import pandas as pd
diag = pd.DataFrame(judge_labels)
print("\n=== rule vs judge ===")
print(diag.head(20))
print("\nRule label distribution:", df['label'].value_counts().to_dict())
print("Judge label distribution:", diag['judge_label'].value_counts().to_dict())
disagreements = (diag['rule_label'] != diag['judge_label']).sum()
print(f"Disagreements: {disagreements}/{len(diag)}")

IndentationError: unexpected indent (1526483860.py, line 2)

## What this validates

- HF login works and gated Gemma weights download (license accepted)
- Tokenization + greedy decode runs without OOM on T4
- FaithEval dataset loads from HF
- Rule-based classifier returns sensible labels
- Claude judge gets called on ambiguous cases and parses correctly
- Anthropic API access works for story generation
- Checkpoint CSV writes

## If this works

Open `m1_extract.ipynb`. Switch `variant='primary'` (Gemma-2-9B-IT), bring up an A100 box.

## If this fails

Fix here before paying for A100 hours. Common failures:
- Gemma license not accepted on HF → 403 on download
- ANTHROPIC_API_KEY missing → story generator + judge crash
- T4 OOM → unlikely on 2B model but reduce `max_new_tokens` if it happens